In [1]:
import pandas as pd
import json
import requests
import numpy as np
from bs4 import BeautifulSoup
from bs4 import XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
import re
import io, os
import csv
import boto3
from botocore.exceptions import ClientError
import openai
from openai import OpenAI
import base64
from PIL import Image
import anthropic
from anthropic import Anthropic

In [8]:
data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover'
ctd_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/CTD_data'
pubtator_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs'
input_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockInput'
output_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockOutput'
figures_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockFigures'
supplement_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockSupplement'
pubtatorFileName = '0-400pub_med_ids.json'

In [3]:
pm_pmc_dict = {'22228119': '3349910', '26714306': '4695093', '28285367': '5387039', '28593498': '5773665', '15154616': '2275409', '17118201': '1676006', '18299717': '2243244', '19114014': '2631511', '25859307': '4387225', '15953866': '2782200', '22110480': '3205718', '22571318': '3485091', '22808131': '3392256', '23724058': '3665791', '24169358': '3833203', '24403256': '3892387', '24413757': '3907846', '24664296': '3963982', '26039251': '4454644', '9118903': '1469763', '17524151': '1890552', '22747577': '3489685', '25351418': '4255228', '25361045': '4245613', '25923732': '4414544'}
pmids = list(pm_pmc_dict.keys())
pmcids = list(pm_pmc_dict.values())

len(pmcids)

25

In [9]:
os.makedirs(pubtator_data_path, exist_ok=True)

pdf_base_url = "https://europepmc.org/backend/ptpmcrender.fcgi"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

for pmid, pmcid in pm_pmc_dict.items():
    pmc_full = f"PMC{pmcid}"
    
    # Download PDF via Europe PMC
    pdf_url = f"{pdf_base_url}?accid={pmc_full}&blobtype=pdf"
    try:
        pdf_response = requests.get(pdf_url, timeout=30, allow_redirects=True, headers=headers)
        pdf_response.raise_for_status()

        if pdf_response.headers.get('Content-Type', '').startswith('application/pdf'):
            pdf_path = os.path.join(pubtator_data_path, f"{pmc_full}.pdf")
            with open(pdf_path, "wb") as f:
                f.write(pdf_response.content)
            print(f"Saved PDF: {pdf_path}")
        else:
            print(f"⚠️ {pmc_full} is not open access or PDF not available — got {pdf_response.headers.get('Content-Type')}")

    except Exception as e:
        print(f"⚠️ Failed PDF for {pmc_full}: {e}")

Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC3349910.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC4695093.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC5387039.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC5773665.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC2275409.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC1676006.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC2243244.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC2631511.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC4387225.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC2782200.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs/PMC3205718.pdf
Saved PDF: /Users/sc/nlp_4_ctd_code/data_js

In [10]:
def createBaselinePromptPDF():

    systemPrompt = """You are an AI assistant that extracts relationships between chemicals and genes from scientific text."""

    mainPrompt = f"""Extract all relationships between chemicals and genes from the text below. Format the output as CSV with one relationship per row.

Columns: "explanation", "chemicalName", "relationship", "geneName", "source"

The relationship must follow this format: "increase <type>", "decrease <type>", or "affect <type>" 
Choose from the following relationship types:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

Output only the CSV with no additional commentary.
"""

    return systemPrompt, mainPrompt

In [17]:
rows = []
for pmid, pmcid in pm_pmc_dict.items():
    pmc_full = f"PMC{pmcid}"
    pdf_path = os.path.join(pubtator_data_path, f"{pmc_full}.pdf")
    
    if os.path.exists(pdf_path):
        with open(pdf_path, "rb") as f:
            system_prompt, main_prompt = createBaselinePromptPDF()
        
        rows.append({
            'pmid': pmid,
            'path': pdf_path,
            'system_prompt': system_prompt,
            'main_prompt': main_prompt
        })
    else:
        print(f"⚠️ PDF not found for PMID {pmid} ({pmc_full})")

pdf_df = pd.DataFrame(rows)
print(f"Found {len(pdf_df)} PDFs")

pdf_df['gpt4.1_output'] = ''
pdf_df

Found 25 PDFs


,pmid,path,system_prompt,main_prompt,gpt4.1_output
0,22228119,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,
1,26714306,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,
2,28285367,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,
3,28593498,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,
4,15154616,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,
5,17118201,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,
6,18299717,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,
7,19114014,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,
8,25859307,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,
9,15953866,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,


In [25]:
gpt_key = 'X'

openai_client = OpenAI(api_key=gpt_key)


def call_gpt(system_prompt, main_prompt, pdf_path=None, model="gpt-4.1"):
    content = []

    if pdf_path and os.path.exists(pdf_path):
        with open(pdf_path, "rb") as f:
            pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")
        content.append({
            "type": "input_file",
            "filename": os.path.basename(pdf_path),
            "file_data": f"data:application/pdf;base64,{pdf_data}"
        })

    content.append({
        "type": "input_text",
        "text": main_prompt
    })

    response = openai_client.responses.create(
        model=model,
        instructions=system_prompt,
        input=[{
            "role": "user",
            "content": content
        }],
        max_output_tokens=10000
    )
    return response.output_text

In [28]:
def run_gpt(df):
    for idx, row in df.iterrows():
        if pd.notna(row.get("gpt4.1_output")):
            existing = row.get("gpt4.1_output")
            if pd.notna(existing) and str(existing).strip() != "":
                continue  # Skip already processed rows
        
        try:
            response = call_gpt(row["system_prompt"], row["main_prompt"], pdf_path=row.get("path"))
            df.at[idx, "gpt4.1_output"] = response
            print(f"done with row {idx}")
        except Exception as e:
            df.at[idx, "gpt4.1_output"] = f"ERROR: {str(e)}"
    
    return df

In [29]:
output_df = run_gpt(pdf_df)

done with row 0
done with row 1
done with row 2
done with row 3
done with row 4
done with row 5
done with row 6
done with row 7
done with row 8
done with row 9
done with row 10
done with row 11
done with row 12
done with row 13
done with row 14
done with row 15
done with row 16
done with row 17
done with row 18
done with row 19
done with row 20
done with row 21
done with row 22
done with row 23
done with row 24


In [30]:
output_df

,pmid,path,system_prompt,main_prompt,gpt4.1_output
0,22228119,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."
1,26714306,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."
2,28285367,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."
3,28593498,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."
4,15154616,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."
5,17118201,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."
6,18299717,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."
7,19114014,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."
8,25859307,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."
9,15953866,/Users/sc/nlp_4_ctd_code/data_json_clover/pubt...,You are an AI assistant that extracts relation...,Extract all relationships between chemicals an...,"explanation,chemicalName,relationship,geneName..."


In [31]:
# output_df.to_csv(f'{output_data_path}/baseline_output_pdf.csv')

In [32]:
def extract_csv(df, source_col="gpt4.1_output"):
    def extract_block(text):
        if pd.isna(text):
            return pd.NA
        
        # Match ```csv ... ``` OR ``` ... ```
        pattern = r"```(?:csv)?\s*(.*?)```"
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        
        if match:
            return match.group(1).strip()
        
        return pd.NA  # No CSV block found

    df["gpt_output"] = df[source_col].apply(extract_block)
    return df

output_df = pd.read_csv(f'{output_data_path}/baseline_output.csv').drop(['Unnamed: 0'], axis=1)
df = extract_csv(output_df)
df

,pmid,pmcid,json_data,system_prompt,main_prompt,gpt4.1_output,gpt_output
0,22228119,PMC3349910,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""explanation"",""chemicalName"",""relation...","""explanation"",""chemicalName"",""relationship"",""g..."
1,26714306,PMC4695093,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""explanation"",""chemicalName"",""relation...","""explanation"",""chemicalName"",""relationship"",""g..."
2,28285367,PMC5387039,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""ascorbic acid decreases expression of...","""ascorbic acid decreases expression of NFκB"",""..."
3,28593498,PMC5773665,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""explanation"",""chemicalName"",""relation...","""explanation"",""chemicalName"",""relationship"",""g..."
4,15154616,PMC2275409,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""explanation"",""chemicalName"",""relation...","""explanation"",""chemicalName"",""relationship"",""g..."
5,17118201,PMC1676006,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""explanation"",""chemicalName"",""relation...","""explanation"",""chemicalName"",""relationship"",""g..."
6,18299717,PMC2243244,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""explanation"",""chemicalName"",""relation...","""explanation"",""chemicalName"",""relationship"",""g..."
7,19114014,PMC2631511,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""explanation"",""chemicalName"",""relation...","""explanation"",""chemicalName"",""relationship"",""g..."
8,25859307,PMC4387225,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""explanation"",""chemicalName"",""relation...","""explanation"",""chemicalName"",""relationship"",""g..."
9,15953866,PMC2782200,"{""bioctype"": ""BioCCollection"", ""source"": ""PMC""...",You are an AI assistant specialized in extract...,### Task: Extract Chemical-Gene Relationships\...,"```csv\n""explanation"",""chemicalName"",""relation...","""explanation"",""chemicalName"",""relationship"",""g..."


In [33]:
def extract_data(df):
    extracted_rows = []

    required_cols = [
        "explanation",
        "chemicalName",
        "relationship",
        "geneName",
        "source"
    ]

    for _, row in df.iterrows():
        pmid = row["pmid"]
        csv_text = row["gpt_output"]

        if pd.isna(csv_text):
            continue

        try:
            temp_df = pd.read_csv(io.StringIO(csv_text))

            # Normalize column names (strip spaces)
            temp_df.columns = [c.strip() for c in temp_df.columns]

            # Add any missing required columns
            for col in required_cols:
                if col not in temp_df.columns:
                    temp_df[col] = pd.NA

            # Keep only required columns (in correct order)
            temp_df = temp_df[required_cols]

            # Add pmid column at front
            temp_df.insert(0, "pmid", pmid)

            extracted_rows.append(temp_df)

        except Exception as e:
            print(f"Error processing PMID {pmid}: {e}")
            continue

    if extracted_rows:
        final_df = pd.concat(extracted_rows, ignore_index=True)
    else:
        final_df = pd.DataFrame(columns=[
            "pmid",
            "explanation",
            "chemicalName",
            "relationship",
            "geneName",
            "source"
        ])

    return final_df

    return final_df

df = extract_data(df)
df

,pmid,explanation,chemicalName,relationship,geneName,source
0,22228119,High concentration of genistein decreases expr...,genistein,decrease expression,activin A,A high concentration of genistein down-regulat...
1,22228119,High concentration of genistein decreases expr...,genistein,decrease expression,Smad3,A high concentration of genistein down-regulat...
2,22228119,High concentration of genistein decreases expr...,genistein,decrease expression,activin B,Downregulation of TGF-β signaling pathway gene...
3,22228119,High concentration of genistein decreases expr...,genistein,decrease expression,Smad3,Downregulation of TGF-β signaling pathway gene...
4,22228119,High concentration of genistein decreases expr...,genistein,decrease expression,TGF-β2,Downregulation of TGF-β signaling pathway gene...
...,...,...,...,...,...,...
322,25923732,CGP57380 decreases interaction (binding) betwe...,CGP57380,decrease binding,eIF4E,Arsenite-induced phosphorylation increased the...
323,25923732,CGP57380 decreases interaction (binding) betwe...,CGP57380,decrease binding,4E-T,Arsenite-induced phosphorylation increased the...
324,25923732,cercosporamide decreases phosphorylation of eIF4E,cercosporamide,decrease phosphorylation,eIF4E,"Indeed, there are already safe and orally avai..."
325,25923732,cercosporamide decreases resistance mediated b...,cercosporamide,decrease response to chemical,eIF4E,"Indeed, there are already safe and orally avai..."


In [34]:
ctd_chemicals = pd.read_csv(f"{ctd_data_path}/CTD_chemicals.csv", skiprows = 27)
ctd_genes = pd.read_csv(f"{ctd_data_path}/CTD_genes.csv", skiprows = 27)

ctd_ixns = pd.read_csv(f"{ctd_data_path}/CTD_chem_gene_ixns.csv", skiprows = 27).drop(labels=0)
ctd_ixns["InteractionActionsList"] = ctd_ixns["InteractionActions"].apply(lambda x: [item.replace("^", " ") for item in x.split("|")])
ctd_ixns["pmcidsList"] = ctd_ixns["PubMedIDs"].apply(lambda x: [str(item) for item in x.split("|")])
ctd_ixns["GeneID"] = ctd_ixns["GeneID"].apply(int)
ctd_ixns["GeneID"] = ctd_ixns["GeneID"].apply(str)
ctd_ixns["ChemicalID"] = ctd_ixns["ChemicalID"].apply(str)

In [42]:
import re
import numpy as np
import pandas as pd
from typing import Optional


# ---------------------------------------------------------------------------
# Small pure helpers
# ---------------------------------------------------------------------------

def _symbol_numeric_suffix(symbol) -> Optional[str]:
    """
    Extract the portion of a GeneSymbol starting from its first digit.
        FKBP1A -> "1A"
        IL6    -> "6"
        TP53   -> "53"
    Returns None if no digit is found or input is NaN.
    """
    if pd.isna(symbol):
        return None
    m = re.search(r"\d.*", str(symbol))
    return m.group(0) if m else None


def _tokenize(name) -> list:
    """Split a (already-cleaned) name string into lowercase tokens."""
    if pd.isna(name) or not str(name).strip():
        return []
    return str(name).strip().lower().split()


def _remove_parentheses(name) -> str:
    """
    Strip parenthesised substrings (including the parentheses themselves).
        "glyceraldehyde-3-phosphate dehydrogenase (GAPDH)"
        -> "glyceraldehyde-3-phosphate dehydrogenase"
    """
    if pd.isna(name):
        return ""
    return re.sub(r"\s*\(.*?\)", "", str(name)).strip()


def _suffix_in_tokens(suffix: Optional[str], tokens: list) -> bool:
    """
    Return True when *suffix* (e.g. "53") is represented in *tokens*.

    Single-token names  : suffix must appear *anywhere* in the token.
    Multi-token names   : at least one token must *end with* suffix.
    """
    if not suffix:
        return False
    suffix = suffix.lower()
    if len(tokens) == 1:
        return suffix in tokens[0]
    return any(tok.endswith(suffix) for tok in tokens)


def _normalize(s) -> str:
    """Lowercase, strip, coerce NaN -> empty string."""
    if pd.isna(s):
        return ""
    return str(s).strip().lower()


def _alt_count(alt_ids_str) -> int:
    """Count non-empty pipe-separated alternative gene IDs."""
    return len([a for a in str(alt_ids_str).split("|") if a.strip()])


def _symbol_is_missing(symbol) -> bool:
    """Return True if a geneSymbol value should be treated as absent."""
    if pd.isna(symbol):
        return True
    return str(symbol).strip() in ("", "-")


def _strip_dot_suffix(symbol) -> Optional[str]:
    """
    If a symbol contains a dot, return the portion before it.
        BRCA1.2  -> "BRCA1"
        TP53.1   -> "TP53"
        BRCA1    -> None  (no dot present)
    """
    if pd.isna(symbol):
        return None
    s = str(symbol).strip()
    if "." in s:
        return s.split(".")[0]
    return None


# ---------------------------------------------------------------------------
# CTD pre-processing  (call once, reuse across batches)
# ---------------------------------------------------------------------------

def prepare_ctd(ctd_genes: pd.DataFrame) -> dict:
    """
    Build fast lookup structures from the CTD gene table.

    Returns a dict with:
        "df"           : cleaned CTD dataframe with helper columns
        "id_index"     : {gene_id_str          -> list[row_index]}
        "symbol_index" : {normalised_symbol    -> list[row_index]}
        "name_index"   : {normalised_gene_name -> list[row_index]}
        "synonym_index": {normalised_synonym   -> list[row_index]}
    """
    ctd = ctd_genes.copy()
    ctd["GeneID"]       = ctd["GeneID"].astype(str)
    ctd["_symbol_norm"] = ctd["# GeneSymbol"].apply(_normalize)
    ctd["_name_norm"]   = ctd["GeneName"].apply(_normalize)
    ctd["_alt_count"]   = ctd["AltGeneIDs"].fillna("").apply(_alt_count)
    ctd["_syns"]        = ctd["Synonyms"].fillna("").str.lower().str.split("|")

    id_index: dict = {}
    for i, gid in ctd["GeneID"].items():
        if gid:
            id_index.setdefault(gid, []).append(i)

    symbol_index: dict = {}
    for i, sym in ctd["_symbol_norm"].items():
        if sym:
            symbol_index.setdefault(sym, []).append(i)

    name_index: dict = {}
    for i, name in ctd["_name_norm"].items():
        if name:
            name_index.setdefault(name, []).append(i)

    synonym_index: dict = {}
    for i, syns in ctd["_syns"].items():
        for s in syns:
            s = s.strip()
            if s:
                synonym_index.setdefault(s, []).append(i)

    return {
        "df": ctd,
        "id_index": id_index,
        "symbol_index": symbol_index,
        "name_index": name_index,
        "synonym_index": synonym_index,
    }


# ---------------------------------------------------------------------------
# Per-row resolution helpers
# ---------------------------------------------------------------------------

def _resolve_by_id(gene_id: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 1: exact GeneID match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["id_index"].get(gene_id, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_symbol(symbol_norm: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 2: exact normalised GeneSymbol match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["symbol_index"].get(symbol_norm, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_name(name_norm: str, name_tokens: list, ctd_lookup: dict) -> Optional[pd.Series]:
    """
    Priority 3: match via GeneName or Synonyms with alt_count tiebreaking.

    Cases
    -----
    1. Only GeneName hits  : prefer suffix-matching candidates; tiebreak by alt_count
    2. Both hits           : compare total alt_count of each group; pick winner's best
    3. Only Synonym hits   : pick highest alt_count
    """
    if not name_norm:
        return None

    ctd_df        = ctd_lookup["df"]
    name_index    = ctd_lookup["name_index"]
    synonym_index = ctd_lookup["synonym_index"]

    gn_rows  = ctd_df.loc[name_index.get(name_norm, [])]
    syn_rows = ctd_df.loc[synonym_index.get(name_norm, [])]

    # Deduplicate: remove synonym hits already present in gene-name hits
    if not gn_rows.empty and not syn_rows.empty:
        syn_rows = syn_rows[~syn_rows.index.isin(gn_rows.index)]

    # ---- Case 1: only GeneName hits ----
    if not gn_rows.empty and syn_rows.empty:
        suffix_matches = [
            r for _, r in gn_rows.iterrows()
            if _suffix_in_tokens(_symbol_numeric_suffix(r["# GeneSymbol"]), name_tokens)
        ]
        best_df = pd.DataFrame(suffix_matches) if suffix_matches else gn_rows
        return best_df.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 2: both hits ----
    if not gn_rows.empty and not syn_rows.empty:
        if gn_rows["_alt_count"].sum() >= syn_rows["_alt_count"].sum():
            return gn_rows.sort_values("_alt_count", ascending=False).iloc[0]
        else:
            return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 3: only Synonym hits ----
    if not syn_rows.empty:
        return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    return None


# ---------------------------------------------------------------------------
# Dot-symbol resolution helper
# ---------------------------------------------------------------------------

def _apply_dot_check(best: pd.Series, ctd_lookup: dict) -> pd.Series:
    """
    If the resolved symbol contains a dot (e.g. BRCA1.2), strip the suffix
    and re-look up the base symbol in CTD to get the correct GeneID/GeneName.
    Returns the corrected row, or the original row with the dot stripped if
    no CTD entry is found for the base.
    """
    base = _strip_dot_suffix(best["# GeneSymbol"])
    if base is None:
        return best  # no dot, nothing to do

    hit = _resolve_by_symbol(_normalize(base), ctd_lookup)
    if hit is not None:
        return hit

    # No CTD entry for base symbol — strip the dot suffix in-place
    best = best.copy()
    best["# GeneSymbol"] = base
    return best


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------
def update_gene_symbol_(
    df: pd.DataFrame,
    ctd_genes: pd.DataFrame,
    ctd_lookup: Optional[dict] = None,
) -> pd.DataFrame:

    df = df.copy()

    df["oldGeneID"]     = df.get("geneID")
    df["oldGeneName"]   = df.get("geneName")
    df["oldGeneSymbol"] = df.get("geneSymbol")

    # ✅ Treat "-", " -", "- " etc. as missing across key gene columns
    for col in ["geneID", "geneSymbol", "geneName"]:
        if col in df.columns:
            df[col] = df[col].replace(r"^\s*-\s*$", "", regex=True).fillna("")

    if ctd_lookup is None:
        ctd_lookup = prepare_ctd(ctd_genes)

    # Vectorised pre-normalisation
    df["_name_norm"] = (
        df.get("geneName", pd.Series(dtype=str))
        .apply(_remove_parentheses)
        .apply(_normalize)
    )
    df["_symbol_norm"] = (
        df.get("geneSymbol", pd.Series(dtype=str))
        .apply(_normalize)
    )

    flags       = np.full(len(df), 2, dtype=int)
    new_symbols = df["geneSymbol"].copy() if "geneSymbol" in df.columns else pd.Series(index=df.index, dtype=str)
    new_ids     = df["geneID"].astype(str).copy()
    new_names   = df["geneName"].copy()   if "geneName"   in df.columns else pd.Series(index=df.index, dtype=str)

    for pos, (idx, row) in enumerate(df.iterrows()):
        raw_symbol  = row.get("geneSymbol")
        gene_id     = str(row.get("geneID", "")).strip()
        symbol_norm = row["_symbol_norm"]
        name_norm   = row["_name_norm"]
        name_tokens = _tokenize(name_norm)

        # Already valid symbol
        if not _symbol_is_missing(raw_symbol):
            if not gene_id:  # symbol present but ID missing — try to resolve ID from symbol
                best = _resolve_by_symbol(symbol_norm, ctd_lookup)
                if best is not None:
                    best = _apply_dot_check(best, ctd_lookup)
                    new_symbols.iloc[pos] = best["# GeneSymbol"]
                    new_ids.iloc[pos]     = best["GeneID"]
                    new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue
    
        # Priority 1: GeneID
        if gene_id:
            best = _resolve_by_id(gene_id, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 2: GeneSymbol
        if symbol_norm:
            best = _resolve_by_symbol(symbol_norm, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 3a: treat geneName as a symbol
        best = _resolve_by_symbol(name_norm, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue

        # Priority 3b: GeneName / Synonym
        best = _resolve_by_name(name_norm, name_tokens, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0

    df["geneSymbol"]     = new_symbols.values
    df["geneID"]         = new_ids.values
    df["geneName"]       = new_names.values
    df["geneSymbolFlag"] = flags

    df.drop(columns=["_name_norm", "_symbol_norm"], inplace=True)
    return df

def build_ctd_chemical_lookup(ctd_chemicals_df):
    chem_id_to_info = {}
    name_to_chem_id = {}

    # Make column names consistent
    ctd_chemicals_df = ctd_chemicals_df.rename(columns=lambda x: x.strip())

    for _, row in ctd_chemicals_df.iterrows():
        chem_name = str(row.get("# ChemicalName", "") or row.get("Chemical Name", "")).strip()
        chem_id = str(row.get("ChemicalID", "") or row.get("Chemical ID", "")).replace("MESH:", "").upper()
        
        # synonyms
        mesh_syn = str(row.get("MESHSynonyms", "")).split("|") if row.get("MESHSynonyms") else []
        ctd_syn = str(row.get("CTDCuratedSynonyms", "")).split("|") if row.get("CTDCuratedSynonyms") else []
        synonyms = [s.strip() for s in mesh_syn + ctd_syn if s.strip()]

        chem_id_to_info[chem_id] = {"chem_name": chem_name, "synonyms": synonyms}

        if chem_name:
            name_to_chem_id[chem_name.upper()] = chem_id

        for syn in synonyms:
            name_to_chem_id[syn.upper()] = chem_id

    return chem_id_to_info, name_to_chem_id



def normalize_gene_id(x):
    """Convert GeneID to clean string or None"""
    if pd.isna(x):
        return None
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return str(x).strip()

def build_ctd_gene_lookups(ctd_genes_df):
    """
    Build lookup dictionaries from CTD genes DataFrame.
    
    Returns:
      geneid_to_name:   {GeneID -> canonical GeneName}
      geneid_to_symbol:{GeneID -> GeneSymbol}
      name_to_geneid:  {UPPERCASE name/symbol/synonym -> GeneID}
      alt_to_geneid:   {AltGeneID -> GeneID}
    """
    geneid_to_name = {}
    geneid_to_symbol = {}
    name_to_geneid = {}
    alt_to_geneid = {}

    for _, row in ctd_genes_df.iterrows():
        gene_id = normalize_gene_id(row.get("GeneID"))
        if not gene_id:
            continue

        symbol = row.get("# GeneSymbol")
        # print(symbol)
        name = row.get("GeneName")
        synonyms = row.get("Synonyms")
        alt_ids = row.get("AltGeneIDs")

        # Canonical name: prefer GeneName over symbol
        canonical_name = name if isinstance(name, str) and name.strip() else symbol
        if not isinstance(canonical_name, str) or not canonical_name.strip():
            continue

        geneid_to_name[gene_id] = canonical_name
        geneid_to_symbol[gene_id] = symbol.strip() if isinstance(symbol, str) else None

        # Index symbol + name
        for n in [symbol, name]:
            if isinstance(n, str) and n.strip():
                name_to_geneid[n.upper()] = gene_id

        # Index synonyms
        if isinstance(synonyms, str) and synonyms.strip():
            for syn in synonyms.split("|"):
                syn = syn.strip()
                if syn:
                    name_to_geneid[syn.upper()] = gene_id

        # Index alternate GeneIDs
        if isinstance(alt_ids, str) and alt_ids.strip():
            for alt in alt_ids.split("|"):
                alt = normalize_gene_id(alt)
                if alt:
                    alt_to_geneid[alt] = gene_id

    return geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid


def update_chemical(df, chem_id_to_info, name_to_chem_id, verbose=True):
    """
    Update chemicalName and chemicalID efficiently using prebuilt lookup tables.
    Flags:
        chemicalFlag = 0 : found normally
        chemicalFlag = 1 : not found
    """
    df = df.copy()
    df['chemicalFlag'] = 0

    # Normalize chemicalID column
    df['chemicalID'] = df['chemicalID'].apply(
        lambda x: str(x).upper().replace("MESH:", "").strip() if pd.notna(x) else None
    )
    df['chemicalName'] = df['chemicalName'].fillna("").astype(str)

    # Precompute normalized name_to_chem_id dict
    norm_name_to_id = {
        k.upper().replace("-", "").replace(" ", ""): v
        for k, v in name_to_chem_id.items()
    }

    for i, row in df.iterrows():
        chem_orig = row['chemicalName']
        chemicalName = chem_orig.strip()
        chemicalID = row['chemicalID']
        chemical_found = False

        # 1️⃣ Exact match by chemicalID
        if chemicalID and chemicalID in chem_id_to_info:
            df.at[i, 'chemicalName'] = chem_id_to_info[chemicalID]['chem_name']
            if verbose:
                print(f"Row {i} - Exact ID match: {chem_id_to_info[chemicalID]['chem_name']}")
            chemical_found = True
            continue  # already found

        # 2️⃣ Exact match by normalized chemicalName
        query_upper = chemicalName.upper().replace("-", "").replace(" ", "")
        if query_upper in norm_name_to_id:
            matched_id = norm_name_to_id[query_upper]
            df.at[i, 'chemicalID'] = matched_id
            df.at[i, 'chemicalName'] = chem_id_to_info[matched_id]['chem_name']
            if verbose:
                print(f"Row {i} - Exact name match: {chem_id_to_info[matched_id]['chem_name']}")
            chemical_found = True
            continue

        # 3️⃣ Partial match
        for key_norm, cid in norm_name_to_id.items():
            if query_upper and query_upper in key_norm:
                df.at[i, 'chemicalID'] = cid
                df.at[i, 'chemicalName'] = chem_id_to_info[cid]['chem_name']
                chemical_found = True
                if verbose:
                    print(f"Row {i} - Partial match: {chem_id_to_info[cid]['chem_name']}")
                break

        # 4️⃣ If still not found
        if not chemical_found:
            df.at[i, 'chemicalFlag'] = 1
            if not df.at[i, 'chemicalName']:
                df.at[i, 'chemicalName'] = chem_orig
            if verbose:
                print(f"Row {i} - Not found in CTD chemicals")
                print(f"  chemicalName: {chem_orig}")
                print(f"  chemicalID:   {chemicalID}")
                print("-" * 40)

    return df

def update_gene(df, geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid, verbose=True):
    """
    Update geneName, geneID, and geneSymbol in the DataFrame using CTD gene lookups.

    Flags:
        geneFlag = 0 : found normally
        geneFlag = 1 : not found

    Additionally:
        oldGeneID = original geneID before normalization/update
    """
    df = df.copy()
    df["geneFlag"] = 0
    
    # ✅ Store original geneID before any modification
    df["oldGeneID"] = df["geneID"]
    df["oldGeneName"] = df["geneName"]
    df["oldGeneSymbol"] = df["geneSymbol"]

    for i, row in df.iterrows():
        original_geneID = row.get("geneID")
        geneID = normalize_gene_id(original_geneID)
        geneName = row.get("geneName")
        geneSymbol = row.get("geneSymbol", None)

        found = False

        # 1️⃣ Resolve by GeneID
        if geneID:
            if geneID in geneid_to_name:
                geneName = geneid_to_name[geneID]
                geneSymbol = geneid_to_symbol.get(geneID)
                found = True

            elif geneID in alt_to_geneid:
                parent_id = alt_to_geneid[geneID]
                geneID = parent_id
                geneName = geneid_to_name.get(parent_id)
                geneSymbol = geneid_to_symbol.get(parent_id)
                found = True
        # somewhere in here, geneSymbol is getting overwritten to geneName.
        
        # 2️⃣ Flags
        if not found:
            df.at[i, "geneFlag"] = 1
            if verbose:
                print(f"Row {i} not found in CTD genes")
                print(f"  geneName: {geneName}")
                print(f"  geneID:   {geneID}")
                print("-" * 40)
        else:
            df.at[i, "geneFlag"] = 0

        # ✅ Overwrite geneID (while oldGeneID is preserved)
        df.at[i, "geneName"] = geneName
        df.at[i, "geneID"] = geneID
        df.at[i, "geneSymbol"] = geneSymbol

    return df


In [43]:
def update_relationship(df, verbose=False):
    import re

    # Allowed CTD relationships
    relationships = [
        "cotreatment", "binding", "reaction", "activity", "localization", "expression", "abundance", 
        "mutagenesis", "response to chemical", "stability", "splicing", "folding", "transport", 
        "uptake", "import", "secretion", "export", "metabolic processing", "chemical synthesis", 
        "degradation", "acetylation", "acylation", "alkylation", "amination", "carbamoylation", 
        "carboxylation", "cleavage", "ethylation", "glycation", "glycosylation", "N-linked glycosylation", 
        "O-linked glycosylation", "glucuronidation", "hydrolysis", "hydroxylation", "lipidation", 
        "geranoylation", "farnesylation", "myristoylation", "palmitoylation", "prenylation", 
        "methylation", "nitrosation", "nucleotidylation", "oxidation", "phosphorylation", "sulfation", 
        "reduction", "ribosylation", "ADP-ribosylation", "ubiquitination", "sumoylation", 
        "glutathionylation", "polymerization"
    ]
    lower_relationships = {r.lower() for r in relationships}

    # Prefix mapping: canonical form
    prefix_map = {
        'increase': 'increases', 'increases': 'increases', 'increased': 'increases', 'increasing': 'increases',
        'decrease': 'decreases', 'decreases': 'decreases', 'decreased': 'decreases', 'decreasing': 'decreases',
        'affect': 'affects', 'affects': 'affects', 'affected': 'affects', 'affecting': 'affects'
    }

    # Prefix exceptions: always "affects"
    affects_required = ['binding', 'cotreatment']

    # Chemical-chemical only relationships
    chem_chem_only = ['abundance', 'response to chemical', 'chemical synthesis']

    # Inhibitory terms replaced with "decrease"
    inhibit_terms = ['inhibit', 'inhibits', 'inhibited', 'inhibiting', 'inhibition']

    df = df.copy()
    df['relationshipWarningFlag'] = 0
    df['prefixMissingFlag'] = 0  # NEW: initialize flag

    if 'relationship' not in df.columns:
        df['relationship'] = ''

    # Normalize text: lowercase, remove quotes, strip spaces
    rel_series = df['relationship'].astype(str).str.lower().str.replace('"', '').str.strip()
    rel_series = rel_series.apply(lambda x: re.sub(r'\s+', ' ', x))

    # Replace inhibitory terms with "decrease"
    for term in inhibit_terms:
        rel_series = rel_series.str.replace(rf'\b{term}\b', 'decrease', regex=True)

    # Normalize prefix and apply rules
    def normalize(rel):
        words = rel.split()
        prefix = words[0] if words and words[0] in prefix_map else None
        core = ' '.join(words[1:]) if prefix else rel

        # NEW: set prefixMissingFlag if no prefix
        if not prefix:
            df.at[df.index[rel_series[rel_series == rel].index[0]], 'prefixMissingFlag'] = 1

        # Prefix exceptions
        if core in affects_required:
            return f"affects {core}"

        # Apply canonical prefix if prefix exists
        if prefix:
            new_prefix = prefix_map.get(prefix, prefix)
            return f"{new_prefix} {core}" if core else new_prefix

        return core

    rel_series = rel_series.apply(normalize)

    # Flag invalid relationships
    def flag_invalid(rel):
        words = rel.split()
        prefix = words[0] if words and words[0] in prefix_map else None
        core = ' '.join(words[1:]) if prefix else rel

        # chem-chem only relationships always invalid
        if core in chem_chem_only:
            return 1

        # Invalid if core not in allowed relationships
        return 0 if core in lower_relationships else 1

    df['relationship'] = rel_series
    df['relationshipWarningFlag'] = rel_series.apply(flag_invalid)

    if verbose:
        invalid_df = df[df['relationshipWarningFlag'] == 1]
        n_invalid = len(invalid_df)
        print(f"{n_invalid} invalid relationships detected.")
        if n_invalid > 0:
            cols_to_show = ['relationship', 'relationshipWarningFlag', 'prefixMissingFlag']
            for col in ['chemicalID', 'geneID']:
                if col in df.columns:
                    cols_to_show.insert(0, col)
            print(invalid_df[cols_to_show].head(20))

    return df

df = update_relationship(df)

,pmid,explanation,chemicalName,relationship,geneName,source,relationshipWarningFlag,prefixMissingFlag,chemicalID
0,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,activin A,A high concentration of genistein down-regulat...,0,0,D019833
1,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,Smad3,A high concentration of genistein down-regulat...,0,0,D019833
2,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,activin B,Downregulation of TGF-β signaling pathway gene...,0,0,D019833
3,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,Smad3,Downregulation of TGF-β signaling pathway gene...,0,0,D019833
4,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,TGF-β2,Downregulation of TGF-β signaling pathway gene...,0,0,D019833
...,...,...,...,...,...,...,...,...,...
322,25923732,CGP57380 decreases interaction (binding) betwe...,CGP57380,affects binding,eIF4E,Arsenite-induced phosphorylation increased the...,0,0,C466997
323,25923732,CGP57380 decreases interaction (binding) betwe...,CGP57380,affects binding,4E-T,Arsenite-induced phosphorylation increased the...,0,0,C466997
324,25923732,cercosporamide decreases phosphorylation of eIF4E,cercosporamide,decreases phosphorylation,eIF4E,"Indeed, there are already safe and orally avai...",0,0,C085452
325,25923732,cercosporamide decreases resistance mediated b...,cercosporamide,decreases response to chemical,eIF4E,"Indeed, there are already safe and orally avai...",1,0,C085452


In [46]:
def normalize_chem_name(name):
    """Normalize chemical names for exact matching"""
    if not isinstance(name, str):
        return ""
    name = name.strip()
    name = name.replace('"', '')
    name = re.sub(r'\s+', ' ', name)
    return name.upper()


def preprocess_for_partial(name):
    """Preprocess chemical name for partial matching"""
    if not isinstance(name, str):
        return ""
    name = re.sub(r'\(.*?\)', '', name)   # remove content inside ()
    name = name.replace('-', '')          # remove dashes
    name = name.strip()
    name = re.sub(r'\s+', ' ', name)
    return name.upper()


def map_chemicals_to_ctd(df, chemicals_df, verbose=False):
    df = df.copy()
    chemicals_df = chemicals_df.fillna("").copy()

    # Lookup dicts for exact match
    name_to_id_main = {}
    name_to_id_mesh = {}
    name_to_id_ctd = {}

    # Store normalized names for partial matching
    partial_lookup = []  # list of tuples (normalized_name, chem_id)

    for _, row in chemicals_df.iterrows():
        chem_id = str(row.get("ChemicalID", "")).strip()
        if chem_id.startswith("MESH:"):
            chem_id = chem_id.replace("MESH:", "")
        chem_id = chem_id.upper()

        # 1️⃣ Main name
        main_name = normalize_chem_name(row.get("# ChemicalName", ""))
        if main_name:
            name_to_id_main[main_name] = chem_id
            partial_lookup.append((preprocess_for_partial(main_name), chem_id))

        # 2️⃣ MESHSynonyms
        for s in row.get("MESHSynonyms", "").split("|"):
            s_norm = normalize_chem_name(s)
            if s_norm:
                name_to_id_mesh[s_norm] = chem_id
                partial_lookup.append((preprocess_for_partial(s_norm), chem_id))

        # 3️⃣ CTDCuratedSynonyms
        for s in row.get("CTDCuratedSynonyms", "").split("|"):
            s_norm = normalize_chem_name(s)
            if s_norm:
                name_to_id_ctd[s_norm] = chem_id
                partial_lookup.append((preprocess_for_partial(s_norm), chem_id))

    chemical_ids = []

    for i, row in df.iterrows():
        chem_name_raw = row.get("chemicalName")

        if not isinstance(chem_name_raw, str) or not chem_name_raw.strip():
            chemical_ids.append("")
            continue

        chem_name_norm = normalize_chem_name(chem_name_raw)

        chem_id = ""

        # ✅ 1️⃣ Exact match (priority order)
        if chem_name_norm in name_to_id_main:
            chem_id = name_to_id_main[chem_name_norm]
        elif chem_name_norm in name_to_id_mesh:
            chem_id = name_to_id_mesh[chem_name_norm]
        elif chem_name_norm in name_to_id_ctd:
            chem_id = name_to_id_ctd[chem_name_norm]

        # ✅ 2️⃣ Partial match (ONLY if exact failed)
        if not chem_id:
            chem_partial = preprocess_for_partial(chem_name_raw)

            for candidate_name, candidate_id in partial_lookup:
                if chem_partial and chem_partial in candidate_name:
                    chem_id = candidate_id
                    break

        chemical_ids.append(chem_id)

        if verbose and not chem_id:
            print(f"Row {i}: '{chem_name_raw}' not found in CTD chemicals")

    df["chemicalID"] = chemical_ids
    return df



df = map_chemicals_to_ctd(df, ctd_chemicals)

In [50]:
df = update_gene_symbol_(df, ctd_genes)
df['geneID'] = ''
df

,pmid,explanation,chemicalName,relationship,geneName,source,relationshipWarningFlag,prefixMissingFlag,chemicalID,geneID,oldGeneID,oldGeneName,oldGeneSymbol,geneSymbol,geneSymbolFlag
0,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,inhibin subunit beta A,A high concentration of genistein down-regulat...,0,0,D019833,,,activin A,None,INHBA,0
1,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,SMAD family member 3,A high concentration of genistein down-regulat...,0,0,D019833,,,Smad3,None,SMAD3,0
2,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,inhibin subunit beta B,Downregulation of TGF-β signaling pathway gene...,0,0,D019833,,,activin B,None,INHBB,0
3,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,SMAD family member 3,Downregulation of TGF-β signaling pathway gene...,0,0,D019833,,,Smad3,None,SMAD3,0
4,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,TGF-β2,Downregulation of TGF-β signaling pathway gene...,0,0,D019833,,,TGF-β2,None,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
322,25923732,CGP57380 decreases interaction (binding) betwe...,CGP57380,affects binding,eukaryotic translation initiation factor 4E,Arsenite-induced phosphorylation increased the...,0,0,C466997,,,eIF4E,None,EIF4E,0
323,25923732,CGP57380 decreases interaction (binding) betwe...,CGP57380,affects binding,eIF4E-Transporter,Arsenite-induced phosphorylation increased the...,0,0,C466997,,,4E-T,None,4E-T,0
324,25923732,cercosporamide decreases phosphorylation of eIF4E,cercosporamide,decreases phosphorylation,eukaryotic translation initiation factor 4E,"Indeed, there are already safe and orally avai...",0,0,C085452,,,eIF4E,None,EIF4E,0
325,25923732,cercosporamide decreases resistance mediated b...,cercosporamide,decreases response to chemical,eukaryotic translation initiation factor 4E,"Indeed, there are already safe and orally avai...",1,0,C085452,,,eIF4E,None,EIF4E,0


In [51]:
def filter_df(df):
    df = df.copy()

    # Convert string "<NA>" to real missing value
    df = df.replace("<NA>", pd.NA)

    # Drop missing
    df = df.dropna(subset=["chemicalID", "geneID"])

    # Ensure flags are numeric (in case they are strings)
    df["relationshipWarningFlag"] = pd.to_numeric(
        df["relationshipWarningFlag"], errors="coerce"
    )
    df["prefixMissingFlag"] = pd.to_numeric(
        df["prefixMissingFlag"], errors="coerce"
    )

    # Keep only rows where both flags == 0
    df = df[
        (df["relationshipWarningFlag"] == 0) &
        (df["prefixMissingFlag"] == 0)
    ]

    return df


df = filter_df(df)
df

,pmid,explanation,chemicalName,relationship,geneName,source,relationshipWarningFlag,prefixMissingFlag,chemicalID,geneID,oldGeneID,oldGeneName,oldGeneSymbol,geneSymbol,geneSymbolFlag
0,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,inhibin subunit beta A,A high concentration of genistein down-regulat...,0,0,D019833,,,activin A,None,INHBA,0
1,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,SMAD family member 3,A high concentration of genistein down-regulat...,0,0,D019833,,,Smad3,None,SMAD3,0
2,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,inhibin subunit beta B,Downregulation of TGF-β signaling pathway gene...,0,0,D019833,,,activin B,None,INHBB,0
3,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,SMAD family member 3,Downregulation of TGF-β signaling pathway gene...,0,0,D019833,,,Smad3,None,SMAD3,0
4,22228119,High concentration of genistein decreases expr...,genistein,decreases expression,TGF-β2,Downregulation of TGF-β signaling pathway gene...,0,0,D019833,,,TGF-β2,None,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,25923732,CGP57380 decreases phosphorylation of eIF4E,CGP57380,decreases phosphorylation,eukaryotic translation initiation factor 4E,we treated cells with the Mnk inhibitor CGP 57...,0,0,C466997,,,eIF4E,None,EIF4E,0
322,25923732,CGP57380 decreases interaction (binding) betwe...,CGP57380,affects binding,eukaryotic translation initiation factor 4E,Arsenite-induced phosphorylation increased the...,0,0,C466997,,,eIF4E,None,EIF4E,0
323,25923732,CGP57380 decreases interaction (binding) betwe...,CGP57380,affects binding,eIF4E-Transporter,Arsenite-induced phosphorylation increased the...,0,0,C466997,,,4E-T,None,4E-T,0
324,25923732,cercosporamide decreases phosphorylation of eIF4E,cercosporamide,decreases phosphorylation,eukaryotic translation initiation factor 4E,"Indeed, there are already safe and orally avai...",0,0,C085452,,,eIF4E,None,EIF4E,0


In [60]:
# ============================================================
# DROP DUPLICATES FUNCTIONS
# ============================================================

def drop_duplicates_(df): # drop duplicates globally with normalized relationship
    pmids = [
        15953866, 17118201, 17524151, 18299717, 19114014, 22110480, 22228119, 22571318,
        22747577, 22808131, 23724058, 24169358, 24403256, 24413757, 24664296, 25351418,
        25361045, 25859307, 25923732, 26039251, 26714306, 28285367, 28593498
    ]
    df = df.copy()
    df = df[df['pmid'].isin(pmids)]
    
    # Normalize relationship (same as in match function)
    def normalize_relationship(s):
        if pd.isna(s):
            return s
        s = str(s).lower().strip()
        s = s.replace("^", " ")
        s = s.replace("increases", "increase")
        s = s.replace("decreases", "decrease")
        s = s.replace("affects", "affect")
        s = " ".join(s.split())
        return s
    
    df['relationship_norm'] = df['relationship'].apply(normalize_relationship)
    
    
    df = df.drop_duplicates(
        subset=['pmid', 'geneSymbol', 'chemicalID', 'relationship_norm'], 
    )
    
    return df



# ============================================================
# MATCH FUNCTION
# ============================================================

def match_manual_and_df(manual: pd.DataFrame, df: pd.DataFrame):

    manual = manual.copy()
    df = df.copy()

    # --------------------------------------------------
    # 1️⃣ Filter manual rows
    # --------------------------------------------------
    if "Chain?" in manual.columns:
        manual = manual[~manual["Chain?"].fillna(False)].reset_index(drop=True)

    # --------------------------------------------------
    # 2️⃣ Rename manual columns
    # --------------------------------------------------
    manual = manual.rename(columns={
        "PMID": "pmid",
        "Interaction Actions": "relationship",
        "Chemical ID": "chemicalID",
        "Gene Symbol": "geneSymbol",
        "Gene ID": "geneID"
    })

    # --------------------------------------------------
    # 3️⃣ Clean PMID
    # --------------------------------------------------
    def clean_pmid(series):
        return (
            series.astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"[^\d]", "", regex=True)
            .replace("", pd.NA)
            .astype("Int64")
        )

    manual["pmid"] = clean_pmid(manual["pmid"])
    df["pmid"] = clean_pmid(df["pmid"])

    # --------------------------------------------------
    # 4️⃣ Normalize chemicalID
    # --------------------------------------------------
    manual["chemicalID"] = (
        manual["chemicalID"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["chemicalID"] = (
        df["chemicalID"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # --------------------------------------------------
    # 5️⃣ Normalize geneID and geneSymbol
    # --------------------------------------------------
    manual["geneID"] = pd.to_numeric(
        manual["geneID"], errors="coerce"
    ).astype("Int64")

    df["geneID"] = pd.to_numeric(
        df["geneID"], errors="coerce"
    ).astype("Int64")

    manual["geneSymbol"] = (
        manual["geneSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["geneSymbol"] = (
        df["geneSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\..+$", "", regex=True)
    )

    # --------------------------------------------------
    # 6️⃣ Build gene_key: prefer geneSymbol, fall back to geneID
    # --------------------------------------------------
    def make_gene_key(row):
        sym = row["geneSymbol"]
        if pd.notna(sym) and sym not in ("", "NAN", "NA"):
            return sym
        if pd.notna(row["geneID"]):
            return str(int(row["geneID"]))
        return pd.NA


    manual["gene_key"] = manual.apply(make_gene_key, axis=1)
    manual["gene_key"] = manual["gene_key"].astype("string")
    
    # For df, only create gene_key if it doesn't exist (from drop_duplicates)
    if 'gene_key' not in df.columns:
        df["gene_key"] = df.apply(make_gene_key, axis=1)
        df["gene_key"] = df["gene_key"].astype("string")
    else:
        df["gene_key"] = df["gene_key"].astype("string")

    # --------------------------------------------------
    # 7️⃣ Normalize relationship
    # --------------------------------------------------
    def normalize_relationship(s):
        if pd.isna(s):
            return s
        s = str(s).lower().strip()
        s = s.replace("^", " ")
        s = s.replace("increases", "increase")
        s = s.replace("decreases", "decrease")
        s = s.replace("affects", "affect")
        s = " ".join(s.split())
        return s

    manual["relationship_norm"] = manual["relationship"].apply(normalize_relationship)
    
    # For df, only create relationship_norm if it doesn't exist (from drop_duplicates)
    if 'relationship_norm' not in df.columns:
        df["relationship_norm"] = df["relationship"].apply(normalize_relationship)

    # --------------------------------------------------
    # 7.5️⃣ Consolidate section types in df
    # --------------------------------------------------
    def consolidate_section(s):
        if pd.isna(s):
            return "OTHER"
        s = str(s).upper()
        if s == "TABLE":
            return "TABLE"
        elif s == "SUPPLEMENT":
            return "SUPPLEMENT"
        elif s in ["FIG", "FIG_TABLE"]:
            return "FIG"
        else:
            return "TEXT"
    
    # Only create section_consolidated if it doesn't exist (from drop_duplicates_section)
    if "section_type" in df.columns and 'section_consolidated' not in df.columns:
        df['section_consolidated'] = df['section_type'].apply(consolidate_section)
    elif "section_type" not in df.columns and 'section_consolidated' not in df.columns:
        df['section_consolidated'] = "UNKNOWN"

    manual = manual.drop_duplicates(
        subset=["pmid", "chemicalID", "relationship", "geneSymbol"]
    ).reset_index(drop=True)

    # --------------------------------------------------
    # 8️⃣ Deduplicate df using merge keys (if not already done)
    # --------------------------------------------------
    merge_keys = ["pmid", "relationship_norm", "chemicalID", "gene_key"]
    
    # Drop duplicates using merge keys
    # df_deduped = df.drop_duplicates(subset=merge_keys)
    df_deduped = df

    # --------------------------------------------------
    # 9️⃣ Overall merge
    # --------------------------------------------------
    matched = manual.merge(
        df_deduped,
        on=merge_keys,
        how="inner",
        suffixes=("_manual", "_df")
    )

    print("=" * 60)
    print("OVERALL MATCHING STATISTICS")
    print("=" * 60)
    print(f"Total matched rows: {len(matched)}")
    match_should = matched[matched["Notes"].astype(str).str.strip().str.lower() == "should"].copy()
    print(f"Length of Should data: {len(match_should)}")
    manual_should = manual[manual["Notes"].astype(str).str.strip().str.lower() == "should"].copy()
    print(f"Length of CTD data: {len(manual)-len(manual_should)}")
    print(f"Overall Precision: {len(matched)/len(df)}")
    print(f"Overall Recall: {len(matched)/(len(manual)-len(manual_should))}")
    
    # --------------------------------------------------
    # 🔟 Section-specific matching
    # --------------------------------------------------
    section_types = ["TABLE", "SUPPLEMENT", "FIG", "TEXT"]
    
    print("\n" + "=" * 60)
    print("SECTION-SPECIFIC MATCHING")
    print("=" * 60)
    
    section_stats = []
    
    for section in section_types:
        # Filter df to this section
        df_section = df_deduped[df_deduped['section_consolidated'] == section]
        
        # Merge manual with this section's df
        matched_section = manual.merge(
            df_section,
            on=merge_keys,
            how="inner",
            suffixes=("_manual", "_df")
        )
         
        section_stats.append({
            'section': section,
            'df_rows': len(df_section),
            'matched_count': len(matched_section),
            'precision': len(matched_section) / len(df_section) if len(df_section) > 0 else 0,
            'recall': len(matched_section) / (len(manual)-len(manual_should)) if (len(manual)-len(manual_should)) > 0 else 0,
        })
        
        print(f"\n{section}:")
        print(f"  DF rows in section: {len(df_section)}")
        print(f"  Matched rows: {len(matched_section)}")
    
    # Summary table
    section_df = pd.DataFrame(section_stats)
    print("\n" + "-" * 60)
    print("SUMMARY TABLE:")
    print(section_df.to_string(index=False))
    print("-" * 60)
    
    # --------------------------------------------------
    # 1️⃣1️⃣ Unmatched rows (overall)
    # --------------------------------------------------
    matched_keys = matched[merge_keys].drop_duplicates(subset=['pmid', 'gene_key', 'chemicalID', 'relationship_norm'])

    unmatched_manual = manual.merge(
        matched_keys,
        on=merge_keys,
        how="left",
        indicator=True
    )
    unmatched_manual = unmatched_manual[
        unmatched_manual["_merge"] == "left_only"
    ].drop(columns="_merge")

    unmatched_df = df.merge(
        matched_keys,
        on=merge_keys,
        how="left",
        indicator=True
    )
    unmatched_df = unmatched_df[
        unmatched_df["_merge"] == "left_only"
    ].drop(columns="_merge")

    print(f"\nUnmatched manual rows: {len(unmatched_manual)}")
    print(f"Unmatched df rows: {len(unmatched_df)}")

    # --------------------------------------------------
    # 1️⃣2️⃣ Partial matches (overall)
    # --------------------------------------------------
    partial_keys = ["pmid", "geneSymbol", "chemicalID"]

    partial_matches = unmatched_manual.merge(
        unmatched_df[partial_keys + ["relationship_norm", "gene_key"]].drop_duplicates(subset=['pmid', 'gene_key', 'chemicalID']),
        on=partial_keys,
        how="inner",
        suffixes=("_manual", "_df")
    )

    if len(partial_matches) > 0:
        print(f"\n⚠️  {len(partial_matches)} rows matched on pmid+geneSymbol+chemicalID but NOT on full merge keys")
    else:
        print("\n✅ No partial matches — unmatched rows don't share pmid+geneSymbol+chemicalID")

    print("=" * 60 + "\n")

    return matched, unmatched_manual, unmatched_df

df = drop_duplicates_(df)
manual = pd.read_csv('/Users/sc/nlp_4_ctd_code/data_json_clover/manual_annotations/CTD.csv')

matched, unmatched_manual, unmatched_df = match_manual_and_df(manual, df)

OVERALL MATCHING STATISTICS
Total matched rows: 58
Length of Should data: 3
Length of CTD data: 609
Overall Precision: 0.27488151658767773
Overall Recall: 0.09523809523809523

SECTION-SPECIFIC MATCHING

TABLE:
  DF rows in section: 0
  Matched rows: 0

SUPPLEMENT:
  DF rows in section: 0
  Matched rows: 0

FIG:
  DF rows in section: 0
  Matched rows: 0

TEXT:
  DF rows in section: 0
  Matched rows: 0

------------------------------------------------------------
SUMMARY TABLE:
   section  df_rows  matched_count  precision  recall
     TABLE        0              0          0     0.0
SUPPLEMENT        0              0          0     0.0
       FIG        0              0          0     0.0
      TEXT        0              0          0     0.0
------------------------------------------------------------

Unmatched manual rows: 571
Unmatched df rows: 153

⚠️  6 rows matched on pmid+geneSymbol+chemicalID but NOT on full merge keys



In [70]:
import os
import pandas as pd
import base64
import zlib
import tiktoken

# -------------------------------
# 1️⃣ Tokenizer setup
# -------------------------------
enc = tiktoken.encoding_for_model("gpt-4o")  # or "gpt-4o-mini" depending on your model

def count_tokens(text):
    """Count tokens safely"""
    if pd.isna(text):
        return 0
    return len(enc.encode(str(text)))

# -------------------------------
# 2️⃣ Process PDFs and compute tokens (compressed)
# -------------------------------
rows = []

for pmid, pmcid in pm_pmc_dict.items():
    pmc_full = f"PMC{pmcid}"
    pdf_path = os.path.join(pubtator_data_path, f"{pmc_full}.pdf")
    
    if os.path.exists(pdf_path):
        with open(pdf_path, "rb") as f:
            pdf_bytes = f.read()
        
        # -------------------------------
        # Compress PDF bytes
        # -------------------------------
        compressed_bytes = zlib.compress(pdf_bytes, level=9)
        pdf_base64 = base64.standard_b64encode(compressed_bytes).decode("utf-8")
        
        # -------------------------------
        # Create system and main prompts
        # -------------------------------
        system_prompt, main_prompt = createBaselinePromptPDF()
        
        # -------------------------------
        # Combine everything for token counting
        # -------------------------------
        full_text = system_prompt + " " + main_prompt + " " + pdf_base64
        token_count = count_tokens(full_text)
        
        rows.append({
            'pmid': pmid,
            'pmcid': pmcid,
            'pdf_path': pdf_path,
            'system_prompt': system_prompt,
            'main_prompt': main_prompt,
            'pdf_base64_compressed': pdf_base64,
            'full_tokens': token_count
        })
    else:
        print(f"⚠️ PDF not found for PMID {pmid} ({pmc_full})")

# -------------------------------
# 3️⃣ Create dataframe
# -------------------------------
pdf_df = pd.DataFrame(rows)
print(f"Found {len(pdf_df)} PDFs")
print(pdf_df[['pmid','full_tokens']].head())

# -------------------------------
# 4️⃣ Optional: summary statistics
# -------------------------------
print("\nToken statistics for all PDFs:")
print(pdf_df['full_tokens'].describe())
print("Average tokens per PDF:", pdf_df['full_tokens'].mean())

Found 25 PDFs
       pmid  full_tokens
0  22228119       413009
1  26714306       258930
2  28285367      1648228
3  28593498       322636
4  15154616       337093

Token statistics for all PDFs:
count    2.500000e+01
mean     1.073118e+06
std      9.640319e+05
min      1.112700e+05
25%      4.138790e+05
50%      6.896180e+05
75%      1.388940e+06
max      4.517873e+06
Name: full_tokens, dtype: float64
Average tokens per PDF: 1073117.96
